In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def typed(name, dtype="string"):
    """Safe cast: blank -> NULL, unparseable -> NULL (never throws). Returns a column named `name`."""
    if dtype == "string":
        return F.col(name).cast("string").alias(name)
    return F.expr(f"try_cast(nullif(trim(cast(`{name}` as string)), '') as {dtype})").alias(name)

bronze_df = spark.table(f"{CATALOG}.bronze.orders")

# Metadata columns (_ingestion_timestamp, _source_file, _pipeline_run_id) are NOT selected, so they are dropped.
# _row_seq remembers the source order so "keep the first occurrence" is deterministic on every rerun.
typed_df = (bronze_df.select(
    typed("o_orderkey", "int"),
    typed("o_custkey", "int"),
    typed("o_orderstatus", "string"),
    typed("o_totalprice", "decimal(18,2)"),
    typed("o_orderdate", "date"),
    typed("o_orderpriority", "string"),
    typed("o_clerk", "string"),
    typed("o_shippriority", "int"),
    typed("o_comment", "string")
    )
    .withColumn("_row_seq", F.monotonically_increasing_id()))

bronze_count = bronze_df.count()
print(f"Bronze orders rows read: {bronze_count}")
display(typed_df.drop("_row_seq").limit(10))

In [0]:
# ── Step 2: Primary Key Validation ──
# Rule: o_orderkey must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK
# First occurrence (by source order) stays in Silver; later repeats are quarantined.

pk_null = F.col("o_orderkey").isNull()
pk_window = Window.partitionBy("o_orderkey").orderBy("_row_seq")      # NO "desc" and NO running count: plain row_number

flagged = (typed_df
    .withColumn("is_null_pk", pk_null)
    .withColumn("_pk_rn", F.when(pk_null, F.lit(0)).otherwise(F.row_number().over(pk_window)))
    .withColumn("is_duplicate_pk", F.col("_pk_rn") > 1))

print(f"NULL_PK failures:      {flagged.filter('is_null_pk').count()}")
print(f"DUPLICATE_PK failures: {flagged.filter('is_duplicate_pk').count()}")

In [0]:
# ── Step 3: Date Range Validation ──
# Rule: the dataset covers a defined historical operating period running
#       from 1 January 1992 to 31 December 1998.
#       o_orderdate records when the order was placed and must fall inside this window.
# A NULL or out-of-window order date is invalid.
# Failure code: INVALID_ORDERDATE

DATE_FROM = "1992-01-01"
DATE_TO   = "1998-12-31"

flagged = flagged.withColumn(
    "is_invalid_orderdate",
    F.col("o_orderdate").isNull()
    | (F.col("o_orderdate") < F.lit(DATE_FROM))
    | (F.col("o_orderdate") > F.lit(DATE_TO))
)

print(f"INVALID_ORDERDATE failures: {flagged.filter('is_invalid_orderdate').count()}")

In [0]:
# ── Step 4: Numeric Range & Domain Validation ──
# Rule: o_totalprice is a monetary value and must be strictly positive (> 0).
# Rule: o_shippriority is a numeric shipping priority and must be >= 0.
# Rule: o_orderpriority must be one of the 5 known TPC-H order priorities.
# Failure codes: INVALID_TOTALPRICE, INVALID_SHIPPRIORITY, INVALID_ORDERPRIORITY

VALID_ORDERPRIORITIES = ["1-URGENT", "2-HIGH", "3-MEDIUM", "4-NOT SPECIFIED", "5-LOW"]

flagged = (flagged
    .withColumn("is_invalid_totalprice",
        F.col("o_totalprice").isNull() | (F.col("o_totalprice") <= 0))
    .withColumn("is_invalid_shippriority",
        F.col("o_shippriority").isNull() | (F.col("o_shippriority") < 0))
    .withColumn("is_invalid_orderpriority",
        ~F.trim(F.col("o_orderpriority")).isin(VALID_ORDERPRIORITIES)))

print(f"INVALID_TOTALPRICE failures:     {flagged.filter('is_invalid_totalprice').count()}")
print(f"INVALID_SHIPPRIORITY failures:    {flagged.filter('is_invalid_shippriority').count()}")
print(f"INVALID_ORDERPRIORITY failures:  {flagged.filter('is_invalid_orderpriority').count()}")

print("\nDistinct order priorities found in bronze (for audit):")
display(bronze_df.select("o_orderpriority").distinct())

In [0]:
# ── Step 5: Referential Integrity Validation ──
# Orders are placed by customers: orders is a CHILD of the customer table.
# The parent is the already-cleaned SILVER customer table (parent-before-child order).
# A NULL or unmatched key is unresolvable -> quarantined.

fk_df = flagged

customer_parent = (spark.table(f"{CATALOG}.silver.customer")
    .select(F.col("c_custkey").alias("_customer_pk")).distinct()
    .withColumn("_customer_hit", F.lit(1)))
fk_df = (fk_df.join(customer_parent, F.col("o_custkey") == F.col("_customer_pk"), "left")
    .withColumn("is_customer", F.col("_customer_hit").isNull())
    .drop("_customer_pk", "_customer_hit"))

flagged = fk_df

print(f"INVALID_FK_CUSTOMER failures: {flagged.filter('is_customer').count()}")

In [0]:
# ── Step 6: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──
flagged = flagged.withColumn(
    "rejection_reason",
    F.concat_ws(", ",
        F.when(F.col("is_null_pk"), F.lit("NULL_PK")),
        F.when(F.col("is_duplicate_pk"), F.lit("DUPLICATE_PK")),
        F.when(F.col("is_invalid_orderdate"), F.lit("INVALID_ORDERDATE")),
        F.when(F.col("is_invalid_totalprice"), F.lit("INVALID_TOTALPRICE")),
        F.when(F.col("is_invalid_shippriority"), F.lit("INVALID_SHIPPRIORITY")),
        F.when(F.col("is_invalid_orderpriority"), F.lit("INVALID_ORDERPRIORITY")),
        F.when(F.col("is_customer"), F.lit("INVALID_FK_CUSTOMER"))
    ),
)

out_cols = ["o_orderkey", "o_custkey", "o_orderstatus", "o_totalprice", "o_orderdate", "o_orderpriority", "o_clerk", "o_shippriority", "o_comment"]

silver_df     = flagged.filter(F.col("rejection_reason") == "").select(*out_cols)
quarantine_df = flagged.filter(F.col("rejection_reason") != "").select(*out_cols, "rejection_reason")

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()
print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# overwrite => rerunning gives the same result
(silver_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.orders"))
(quarantine_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.orders_quarantine"))

print("Tables written: silver.orders, silver.orders_quarantine")

In [0]:
# ── Step 7: Bronze-Silver-Quarantine Reconciliation ──
bronze_total     = spark.table(f"{CATALOG}.bronze.orders").count()
silver_total     = spark.table(f"{CATALOG}.silver.orders").count()
quarantine_total = spark.table(f"{CATALOG}.silver.orders_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != Silver ({silver_total}) + Quarantine ({quarantine_total})"
)
print("Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 8: Data Quality Report ──
# Every check is listed, even when zero rows fail.
rules = [
    ("NULL_PK", "o_orderkey must not be null", "is_null_pk"),
    ("DUPLICATE_PK", "o_orderkey must be unique (first occurrence kept)", "is_duplicate_pk"),
    ("INVALID_ORDERDATE", f"o_orderdate must be within {DATE_FROM} .. {DATE_TO}", "is_invalid_orderdate"),
    ("INVALID_TOTALPRICE", "o_totalprice must be strictly positive (> 0)", "is_invalid_totalprice"),
    ("INVALID_SHIPPRIORITY", "o_shippriority must be >= 0", "is_invalid_shippriority"),
    ("INVALID_ORDERPRIORITY", f"o_orderpriority must be one of {', '.join(VALID_ORDERPRIORITIES)}", "is_invalid_orderpriority"),
    ("INVALID_FK_CUSTOMER", "o_custkey must exist in silver.customer", "is_customer")
]
dq_rows = [("orders", code, desc, flagged.filter(F.col(flag)).count(), "Quarantine") for code, desc, flag in rules]

dq_report = spark.createDataFrame(
    dq_rows, ["table", "rule_code", "check_description", "failing_rows", "action_taken"]
)
display(dq_report)

# Persist so the whole run is auditable (overwrite per table is not needed: one table per notebook)
(dq_report.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dq_report_orders"))

if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table(f"{CATALOG}.silver.orders_quarantine").limit(10))
else:
    print("\nNo quarantined records — all rows passed every check.")

print(f"\n── Silver orders ({silver_count} rows) ──")
display(spark.table(f"{CATALOG}.silver.orders").limit(20))